# PNADc Historical Proxy Engine v1.0.1\nCorreção: seleção de layout regular separada do seletor dos suplementos diretos.\n

# SPINE-GPE v7 — PNADc Historical Certification & Proxy Calibration Engine v1.0.1

Este notebook executa a auditoria, a calibração temporal 2022T4/2024T3 e a certificação das fontes regulares da PNADc já disponíveis no Drive.

**Limite obrigatório:** a série histórica recebe probabilidade modelada (`evidence tier C`); `platform_delivery_direct` permanece ausente.


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPT = ROOT / 'scripts/SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.py'
REQ = ROOT / 'scripts/requirements_SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.txt'
UPSTREAM = ROOT / 'scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py'

print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT, SCRIPT.exists())
print('REQ:', REQ, REQ.exists())
print('UPSTREAM:', UPSTREAM, UPSTREAM.exists())
assert SCRIPT.exists()
assert REQ.exists()
assert UPSTREAM.exists()


ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.py True
REQ: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/requirements_SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.txt True
UPSTREAM: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py True


In [7]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)
subprocess.run([sys.executable, '-m', 'py_compile', str(SCRIPT)], check=True)
print('py_compile: OK')


py_compile: OK


## 1. Auditoria das fontes locais

O modo `auto` descobre os trimestres já materializados em `data_pnadc` e `01_raw/10_ibge/pnadc_historical`, excluindo os módulos especiais diretos.


In [8]:
audit = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'audit',
        '--periods', 'auto',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(audit.stdout)
print(audit.stderr)
print('Audit exit code:', audit.returncode)


2026-07-23 12:30:16,271 | INFO | SPINE-GPE PNADc Historical Proxy Engine v1.0.1 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=audit | periods=auto
2026-07-23 12:35:47,502 | INFO | Auditoria concluída | status=AUDIT_PASSED | períodos=['2019q1', '2019q2', '2019q3', '2019q4', '2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q4', '2024q3']
{
  "run_id": "20260723T123016Z",
  "script_version": "1.0.1",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.1",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "requested_periods": [
    "2019q1",
    "2019q2",
    "2019q3",
    "2019q4",
    "2020q1",
    "2020q2",
    "2020q3",
    "2020q4",
    "2021q1",
    "2021q2",
    "2021q3",
    "2021q4",
    "2022q4",
    "2024q3"
  ],
  "sources": [
    {
      "period": "2019q1",
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/pnadc_histor

In [9]:
AUDIT_LOCK = ROOT / '00_admin/PNADC_HISTORICAL_PROXY_AUDIT_LOCK.json'
if not AUDIT_LOCK.exists():
    raise RuntimeError('Lock de auditoria não foi criado. Revise STDOUT/STDERR.')
audit_lock = json.loads(AUDIT_LOCK.read_text(encoding='utf-8'))
print(json.dumps(audit_lock, ensure_ascii=False, indent=2))
assert audit_lock['status'] == 'AUDIT_PASSED', audit_lock['critical_failures']
print('Períodos descobertos:', audit_lock['requested_periods'])


{
  "run_id": "20260723T123016Z",
  "script_version": "1.0.1",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.1",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "requested_periods": [
    "2019q1",
    "2019q2",
    "2019q3",
    "2019q4",
    "2020q1",
    "2020q2",
    "2020q3",
    "2020q4",
    "2021q1",
    "2021q2",
    "2021q3",
    "2021q4",
    "2022q4",
    "2024q3"
  ],
  "sources": [
    {
      "period": "2019q1",
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/pnadc_historical/2019q1/PNADC_012019.txt",
      "suffix": ".txt",
      "sha256": "ac864079cd089a770c756b01f89a606a8e56803b99f656ddf046b67f9e477f3d",
      "record_width": 3478,
      "layout_path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/dicionario_PNADC_microdados_trimestral.xls",
      "layout_sha256": "3f6b46e

## 2. Calibração e certificação

Esta etapa:

1. lê os Parquets diretos certificados de 2022 e 2024;
2. valida temporalmente o modelo nos dois sentidos;
3. calibra a probabilidade;
4. aplica o modelo apenas aos trimestres regulares descobertos;
5. gera outputs imutáveis, model card, estimativas, lock e freeze.


In [10]:
full = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'full',
        '--periods', 'auto',
        '--chunk-rows', '50000',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(full.stdout)
print(full.stderr)
print('Full exit code:', full.returncode)


2026-07-23 12:40:00,706 | INFO | SPINE-GPE PNADc Historical Proxy Engine v1.0.1 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=full | periods=auto
2026-07-23 12:41:50,022 | INFO | 2019q1: 50000 registros brutos lidos; 16621 no universo histórico
2026-07-23 12:41:59,394 | INFO | 2019q1: 100000 registros brutos lidos; 30984 no universo histórico
2026-07-23 12:42:06,636 | INFO | 2019q1: 150000 registros brutos lidos; 46059 no universo histórico
2026-07-23 12:42:16,184 | INFO | 2019q1: 200000 registros brutos lidos; 61493 no universo histórico
2026-07-23 12:42:23,417 | INFO | 2019q1: 250000 registros brutos lidos; 75789 no universo histórico
2026-07-23 12:42:32,997 | INFO | 2019q1: 300000 registros brutos lidos; 94937 no universo histórico
2026-07-23 12:42:41,929 | INFO | 2019q1: 350000 registros brutos lidos; 114775 no universo histórico
2026-07-23 12:42:50,403 | INFO | 2019q1: 400000 registros brutos lidos; 135682 no universo histórico
2026-07-23 12:43:00,176 | INFO 

In [11]:
LOCK = ROOT / '00_admin/PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json'
if not LOCK.exists():
    raise RuntimeError('Lock de certificação não foi criado. Revise STDOUT/STDERR.')
lock = json.loads(LOCK.read_text(encoding='utf-8'))
print(json.dumps(lock, ensure_ascii=False, indent=2))
assert lock['status'] == 'CERTIFIED', lock['critical_failures']
print('STATUS:', lock['status'])
print('MODELO:', lock['model'])
print('PERÍODOS:', lock['certified_periods'])
print('REPORT:', lock['report'])


{
  "run_id": "20260723T124000Z",
  "script_version": "1.0.1",
  "schema_version": "spine-gpe-v7-pnadc-historical-proxy-1.0.0",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.1",
  "model_schema_version": "spine-gpe-v7-pnadc-proxy-model-1.0.0",
  "mode": "full",
  "status": "CERTIFIED",
  "critical_failures": [],
  "warnings": [],
  "requested_periods": [
    "2019q1",
    "2019q2",
    "2019q3",
    "2019q4",
    "2020q1",
    "2020q2",
    "2020q3",
    "2020q4",
    "2021q1",
    "2021q2",
    "2021q3",
    "2021q4",
    "2022q4",
    "2024q3"
  ],
  "certified_periods": [
    "2019q1",
    "2019q2",
    "2019q3",
    "2019q4",
    "2020q1",
    "2020q2",
    "2020q3",
    "2020q4",
    "2021q1",
    "2021q2",
    "2021q3",
    "2021q4",
    "2022q4",
    "2024q3"
  ],
  "direct_inputs": {
    "2022": {
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/10_pnadc_certified/certified_pnadc_platform_2022.parquet",
      "sha2

## 3. Inspeção das métricas temporais e estimativas


In [12]:
import pandas as pd

run_id = lock['run_id']
metrics_path = ROOT / f'05_outputs/tables/pnadc_historical_proxy/pnadc_proxy_temporal_metrics_{run_id}.csv'
rules_path = ROOT / f'05_outputs/tables/pnadc_historical_proxy/pnadc_proxy_rule_benchmark_{run_id}.csv'
estimates_path = Path(lock['estimates'])

metrics = pd.read_csv(metrics_path)
rules = pd.read_csv(rules_path)
estimates = pd.read_csv(estimates_path)

display(metrics)
display(rules)
display(estimates)


,model,train_year,test_year,n_test,n_positive,weighted_prevalence,roc_auc,average_precision,brier,null_brier,brier_skill,log_loss,ece_10
0,weighted_logit_occ_activity_position,2024,2022,178163,691,0.005208,0.980863,0.331738,0.003954,0.005181,0.236815,0.016074,5.287540e-04
1,weighted_logit_occ_activity_position,2022,2024,184157,778,0.005505,0.980062,0.374635,0.003986,0.005475,0.271932,0.016302,6.793733e-04
2,weighted_logit_occ_activity_position_calibrate...,0,0,362320,1469,0.005359,0.981253,0.349444,0.003916,0.005331,0.265323,0.015840,1.172943e-18
3,weighted_logit_extended_sensitivity,2024,2022,178163,691,0.005208,0.982918,0.372392,0.003912,0.005181,0.245040,0.015604,5.297994e-04
4,weighted_logit_extended_sensitivity,2022,2024,184157,778,0.005505,0.981921,0.389540,0.003968,0.005475,0.275281,0.015932,5.507342e-04
5,weighted_logit_extended_sensitivity_calibrated...,0,0,362320,1469,0.005359,0.982972,0.377126,0.003876,0.005331,0.272829,0.015465,2.150017e-18


,year,rule,n,n_positive,weighted_sensitivity,weighted_specificity,weighted_ppv,weighted_npv,weighted_accuracy
0,2022,occupation_compatible,178163,691,0.531010,0.938692,0.043379,0.997391,0.936568
1,2022,delivery_activity,178163,691,0.431591,0.997109,0.438705,0.997024,0.994164
2,2022,occupation_or_activity,178163,691,0.535719,0.938413,0.043557,0.997416,0.936316
3,2022,occupation_and_activity,178163,691,0.426882,0.997388,0.461083,0.997001,0.994416
4,2024,occupation_compatible,184157,778,0.561790,0.938108,0.047843,0.997421,0.936036
5,2024,delivery_activity,184157,778,0.459623,0.997019,0.460521,0.997009,0.994061
6,2024,occupation_or_activity,184157,778,0.563402,0.937908,0.047827,0.997430,0.935846
7,2024,occupation_and_activity,184157,778,0.458011,0.997219,0.476942,0.997000,0.994251


,period,estimand,total,total_se,share,share_se,n,n_eff,n_strata,n_psu
0,2019q1,model_expected_probability,3.162166e+05,7404.902800,0.003942,0.000091,201434,101639.770957,574,15060
1,2019q1,rule_occupation,4.769847e+06,66522.418921,0.059459,0.000775,201434,101639.770957,574,15060
2,2019q1,rule_activity,1.482716e+05,12933.224719,0.001848,0.000161,201434,101639.770957,574,15060
3,2019q1,rule_occ_or_activity,4.776935e+06,66597.844060,0.059547,0.000776,201434,101639.770957,574,15060
4,2019q1,rule_occ_and_activity,1.411835e+05,12593.250210,0.001760,0.000157,201434,101639.770957,574,15060
...,...,...,...,...,...,...,...,...,...,...
107,2024q3,rule_occ_or_activity,5.740229e+06,82087.275655,0.064852,0.000860,184157,95984.942282,573,15070
108,2024q3,rule_occ_and_activity,4.679431e+05,24295.117989,0.005287,0.000273,184157,95984.942282,573,15070
109,2024q3,model_class_threshold_0.25,3.933160e+05,22746.793421,0.004444,0.000256,184157,95984.942282,573,15070
110,2024q3,model_class_threshold_0.50,3.233188e+05,20759.682765,0.003653,0.000234,184157,95984.942282,573,15070


## 4. Expansão histórica opcional

Execute somente depois que o modo `auto` estiver certificado. O comando abaixo pode baixar e processar muitos arquivos grandes.


In [13]:
# Exemplo controlado: período pandêmico e pré-módulo direto.

expansion = subprocess.run(
     [
         sys.executable, str(SCRIPT),
         '--root', str(ROOT),
         '--mode', 'full',
         '--periods', '2019q1:2021q4',
         '--download-missing',
         '--chunk-rows', '50000',
         '--strict',
     ],
     check=False,
 )


## Regras de uso

- A soma ponderada das probabilidades é o estimando principal.
- A classe binária é somente análise de sensibilidade.
- A série histórica não preenche `SD14001`, `S140093` ou `platform_delivery_direct`.
- Comparações entre períodos são descritivas/model-based, não um desenho causal.
